# Fine-Tuning an LLM to Translate Human Text to SQL
In this lab, I will be fine tuning the Qwen3 4B model on data that consists of human instructions and their corresponding SQL queries. My hope is that I will be able to have a model that does exceptional at converting language to SQL so that others who are not SQL proficient can perform tasks that use it.

## Setting up and importing our dependencies

In [1]:
# Mounting Google Drive so we can save model checkpoints

from google.colab import drive
import os

drive.mount('/content/drive')
output_dir = "/content/drive/MyDrive/sql-fine-tuning"
os.makedirs(output_dir, exist_ok=True)

Mounted at /content/drive


In [2]:
# Installing and Importing Our Dependencies

!pip install -U trl
!pip install accelerate
!pip install -U torchao
!pip install -U bitsandbytes>=0.46.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
import os
import sqlite3
import pandas
import re
import torch
import peft
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset

## Dataset Description

The dataset that I will be using is the filtered version of the famous BIRD-SQL dataset that is often used to fine-tune models. While it is a filtered version of the original, attempts to fine-tune with this dataset has produced results almost exactly the same as the original.

The dataset contains around 6600 JSON entries with four parts

* db_id: The database name
* question: The question that is asked by the human
* evidence: external knowledge
* SQL: The query that answers the human's question.


In [4]:
# Load the dataset
dataset = load_dataset("birdsql/bird_mini_dev")
# Access the dataset
dataset["mini_dev_sqlite"][0]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

mini_dev_mysql-00000-of-00001.json:   0%|          | 0.00/307k [00:00<?, ?B/s]

mini_dev_pg-00000-of-00001.json:   0%|          | 0.00/284k [00:00<?, ?B/s]

mini_dev_sqlite-00000-of-00001.json:   0%|          | 0.00/279k [00:00<?, ?B/s]

Generating mini_dev_mysql split: 0 examples [00:00, ? examples/s]

Generating mini_dev_pg split: 0 examples [00:00, ? examples/s]

Generating mini_dev_sqlite split: 0 examples [00:00, ? examples/s]

{'question_id': 1471,
 'db_id': 'debit_card_specializing',
 'question': 'What is the ratio of customers who pay in EUR against customers who pay in CZK?',
 'evidence': "ratio of customers who pay in EUR against customers who pay in CZK = count(Currency = 'EUR') / count(Currency = 'CZK').",
 'SQL': "SELECT CAST(SUM(IIF(Currency = 'EUR', 1, 0)) AS FLOAT) / SUM(IIF(Currency = 'CZK', 1, 0)) AS ratio FROM customers",
 'difficulty': 'simple'}

In [ ]:
dataset

## Loading in the Model and Testing it.

Here we are going to load in the Qwen3 4B Model in. I chose this model because Qwen models are strong at generating structured output and the 4B parameter model is small enough to feasibly train within this Colab environment while also providing excellent results.

Before I apply LoRA, I want to see how the base model performs on its own.

In [ ]:
# Quantization so that the model doesn't take up as much memory
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model_id = "Qwen/Qwen3-4B"
model = AutoModelForCausalLM.from_pretrained(model_id,
                                             torch_dtype="auto",
                                             quantization_config=bnb_config,
                                             device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)
print(model.dtype)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

torch.bfloat16


In [5]:

# This function takes in the name of the database and gets its schema from the BIRD database
def get_table_schema(db_id):
  base_path = "/content/drive/MyDrive/sql-fine-tuning/dev_databases"
  db_path = os.path.join(base_path, db_id, f"{db_id}.sqlite")

  if not os.path.exists(db_path):
    return f"Schema context unavailable for {db_id}."

  conn = sqlite3.connect(db_path)
  cursor = conn.cursor()

  cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';")
  tables = [row[0] for row in cursor.fetchall()]

  schema_parts = []

  for table in tables:
    cursor.execute(f"PRAGMA table_info(`{table}`);")
    columns = [col[1] for col in cursor.fetchall()]
    columns_str = ", ".join(columns)
    schema_parts.append(f"{table}({columns_str})")

  conn.close()
  return " | ".join(schema_parts)

In [ ]:
schema = get_table_schema("california_schools")
schema

'frpm(CDSCode, Academic Year, County Code, District Code, School Code, County Name, District Name, School Name, District Type, School Type, Educational Option Type, NSLP Provision Status, Charter School (Y/N), Charter School Number, Charter Funding Type, IRC, Low Grade, High Grade, Enrollment (K-12), Free Meal Count (K-12), Percent (%) Eligible Free (K-12), FRPM Count (K-12), Percent (%) Eligible FRPM (K-12), Enrollment (Ages 5-17), Free Meal Count (Ages 5-17), Percent (%) Eligible Free (Ages 5-17), FRPM Count (Ages 5-17), Percent (%) Eligible FRPM (Ages 5-17), 2013-14 CALPADS Fall 1 Certification Status) | satscores(cds, rtype, sname, dname, cname, enroll12, NumTstTakr, AvgScrRead, AvgScrMath, AvgScrWrite, NumGE1500) | schools(CDSCode, NCESDist, NCESSchool, StatusType, County, District, School, Street, StreetAbr, City, Zip, State, MailStreet, MailStrAbr, MailCity, MailZip, MailState, Phone, Ext, Website, OpenDate, ClosedDate, Charter, CharterNum, FundingType, DOC, DOCType, SOC, SOCTyp

In [6]:
# Creating a Function that formats the JSON entries into a sentence for the model
# When we traing it

def format(example):
  messages = [
  {
    "role": "system",
    "content": """
    You are an expert Text-to-SQL model.

    Return ONLY valid SQLite SQL.
    Do not explain your reasoning.
    Do not use markdown.
    Never invent columns not in schema.
    """
  },
  {
    "role": "user",
    "content": f"""Convert the question into SQL.

  Schema:
  {get_table_schema(example['db_id'])}

  Question:
  {example['question']}

  Evidence:
  {example["evidence"]}"""
  },
  {
    "role": "assistant",
    "content": example["SQL"]
  }
    ]

  return tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True,
      enable_thinking=False
  )

In [7]:
# This function formats the JSON entry for evaluation, here we don't include
# the SQL answer as we don't want the model to know the answer

def format_eval(example):
  messages = [
  {
    "role": "system",
    "content": """
    You are an expert Text-to-SQL model.

    Return ONLY valid SQLite SQL.
    Do not explain your reasoning.
    Do not use markdown.
    Never invent columns not in schema.
    """
  },
  {
    "role": "user",
    "content": f"""Convert the question into SQL.

  Schema:
  {get_table_schema(example['db_id'])}

  Question:
  {example['question']}

  Evidence:
  {example["evidence"]}"""
  },
    ]

  return tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True,
      enable_thinking=False
  )

In [ ]:
prompt = format_eval(dataset["mini_dev_sqlite"][338])
print(prompt)
print("\n")
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
  outputs = model.generate(
      **inputs,
      max_new_tokens=256,
      temperature=0.7,
      do_sample=True
  )
generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
base_output = tokenizer.decode(generated_ids, skip_special_tokens=True)
print(base_output)

<|im_start|>system

    You are an expert Text-to-SQL model.

    Return ONLY valid SQLite SQL.
    Do not explain your reasoning.
    Do not use markdown.
    Never invent columns not in schema.
    <|im_end|>
<|im_start|>user
Convert the question into SQL.

  Schema:
  badges(Id, UserId, Name, Date) | comments(Id, PostId, Score, Text, CreationDate, UserId, UserDisplayName) | postHistory(Id, PostHistoryTypeId, PostId, RevisionGUID, CreationDate, UserId, Text, Comment, UserDisplayName) | postLinks(Id, CreationDate, PostId, RelatedPostId, LinkTypeId) | posts(Id, PostTypeId, AcceptedAnswerId, CreaionDate, Score, ViewCount, Body, OwnerUserId, LasActivityDate, Title, Tags, AnswerCount, CommentCount, FavoriteCount, LastEditorUserId, LastEditDate, CommunityOwnedDate, ParentId, ClosedDate, OwnerDisplayName, LastEditorDisplayName) | tags(Id, TagName, Count, ExcerptPostId, WikiPostId) | users(Id, Reputation, CreationDate, DisplayName, LastAccessDate, WebsiteUrl, Location, AboutMe, Views, UpVote

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


SELECT COUNT(*) FROM comments WHERE PostId = (SELECT PostId FROM posts ORDER BY Score DESC LIMIT 1);


In [8]:
# sqlite connections cache to reduce overhead

connections = {}

def get_connection(db_path):
    if db_path not in connections:
        connections[db_path] = sqlite3.connect(db_path)
    return connections[db_path]

In [9]:
# Function to only extract the first sql query the model generates if the model generates 2

def extract_first_sql(sql):
    if ";" in sql:
        return sql.split(";")[0].strip() + ";"
    return sql.strip()

# Function to normalize the sql query to compare them
def normalize_sql(sql):
    return sql.strip().rstrip(";").lower()

In [10]:
# Evaluating the base model based on Execution accuracy

from tqdm import tqdm
import time

def determine_error_type(e, error_type):
  if "interrupted" in e:
    error_type["timeouts"] += 1
  elif "no such column" in e:
    error_type['no_column'] += 1
  elif "misuse" in e:
    error_type["misuse"] += 1
  elif "syntax error" in e:
    error_type['syntax'] += 1

In [ ]:
# Examining the distribution on token count for the SQL targets. This will help me choose
# what max_new_tokens should be
import numpy as np
lengths = []

for sample in dataset["mini_dev_sqlite"]:
    sql = sample["SQL"]
    lengths.append(len(tokenizer.encode(sql, add_special_tokens=False)))
print("Mean:", np.mean(lengths))
print("Median:", np.median(lengths))
print("90th:", np.percentile(lengths, 90))
print("95th:", np.percentile(lengths, 95))
print("99th:", np.percentile(lengths, 99))
print("Max:", np.max(lengths))

Mean: 63.034
Median: 59.5
90th: 89.10000000000002
95th: 106.04999999999995
99th: 209.35999999999967
Max: 480


In [11]:
def eval_model(model):
  base_path = "/content/drive/MyDrive/sql-fine-tuning/dev_databases"
  correct_output = 0
  exact_match = 0
  sql_errors = 0
  error_type = {
      "syntax": 0,
      "no_column": 0,
      "misuse": 0,
      "timeouts": 0,
  }

  for i, sample in enumerate(tqdm(dataset["mini_dev_sqlite"])):
    print(f"Starting sample {i} , {sample["db_id"]}")
    prompt = format_eval(sample)
    db_id = sample["db_id"]
    gold_sql = sample['SQL']
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
      outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
      )
    print("Finished Generating")
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    pred_sql = tokenizer.decode(generated_ids, skip_special_tokens=True)
    pred_sql = extract_first_sql(pred_sql)
    db_path = os.path.join(base_path, db_id, f"{db_id}.sqlite")
    conn = get_connection(db_path)
    cursor = conn.cursor()

    start_time = time.time()

    def progress_handler():
      if time.time() - start_time > 10:
        return 1
      return 0

    conn.set_progress_handler(progress_handler, 10000)

    try:
      gold_result = cursor.execute(gold_sql).fetchall()

      pred_result = cursor.execute(pred_sql).fetchall()

      if gold_result == pred_result:
        correct_output += 1
      if normalize_sql(gold_sql) == normalize_sql(pred_sql):
        exact_match += 1
    except sqlite3.OperationalError as e:
      determine_error_type(str(e), error_type)
      sql_errors += 1
      print(f"Sample {i}: SQLite error: {e}")
    except Exception as e:
      print(f"Unexpected error on sample {i}: {e}")
      raise
    finally:
      conn.set_progress_handler(None, 0)
    print("Finished Execution\n")

  for path in connections:
    connections[path].close()

  correct_output /= len(dataset["mini_dev_sqlite"])
  exact_match /= len(dataset["mini_dev_sqlite"])
  sql_errors /= len(dataset["mini_dev_sqlite"])
  return {"correct_output" : correct_output, "exact_sql" : exact_match, "sql_errors": sql_errors}, error_type


In [ ]:
base_results = eval_model(model)

  0%|          | 0/500 [00:00<?, ?it/s]

Starting sample 0 , debit_card_specializing


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
  0%|          | 1/500 [00:05<46:49,  5.63s/it]

Finished Generating
Finished Execution

Starting sample 1 , debit_card_specializing
Finished Generating


  0%|          | 2/500 [00:20<1:31:28, 11.02s/it]

...

Finished Generating
Sample 497: SQLite error: no such function: DATEDIFF
Finished Execution

Starting sample 498 , financial
Finished Generating


100%|█████████▉| 499/500 [51:00<00:09,  9.61s/it]

Sample 498: SQLite error: interrupted
Finished Execution

Starting sample 499 , financial


100%|██████████| 500/500 [51:07<00:00,  6.14s/it]

Finished Generating
Finished Execution



In [ ]:
print(base_results[0])
print(base_results[1])

{'correct_output': 0.228, 'exact_sql': 0.034, 'sql_errors': 0.378}
{'syntax': 25, 'no_column': 121, 'misuse': 8, 'timeouts': 9}


## Training the Model with LoRA

Now that we have evaluated the base model, we can see that it can use a alot of improvement. To do this we will fine-tune with Low-Rank Adaptation (LoRA) which is a technique that freezes the model weights and trains smaller matrices that will learn the new concepts. That way we dont have to train all the 4 Billion weights

In [14]:
from peft import LoraConfig, prepare_model_for_kbit_training

for name, module in model.named_modules():
  print(name)

In [ ]:
# We will be applying LoRA to all the Attention Layers and MLP layers

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    task_type="CAUSAL_LM"
)

In [ ]:
model = prepare_model_for_kbit_training(model)


In [ ]:
split = dataset['mini_dev_sqlite'].train_test_split(test_size=0.2, seed=42)

train_dataset = split["train"]
test_dataset = split["test"]

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/sql-fine-tuning",
    num_train_epochs=4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    eval_strategy="epoch",
    fp16=False,
    bf16=True
)

In [ ]:
trainer = SFTTrainer(model=model,
                     train_dataset=train_dataset,
                     eval_dataset=test_dataset,
                     args=training_args,
                     peft_config=lora_config,
                     formatting_func=lambda x: format(x))
trainer.train(resume_from_checkpoint=True)

Applying formatting function to train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
3,0.177692,0.210322,0.190500,183891.000000,0.950933


## Evaluating our fine-tuned model

Now that we have trained the LoRA adapter, it is time to evaluate it and compare it to the base model

In [16]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
model_id = "Qwen/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(model_id)

base_model =  AutoModelForCausalLM.from_pretrained(model_id,
                                             torch_dtype="auto",
                                             quantization_config=bnb_config,
                                             device_map="auto")
lora_model = PeftModel.from_pretrained(
    base_model,
    "/content/drive/MyDrive/sql-fine-tuning/checkpoint-200"
)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [17]:
# Evaluating the fine-tuned model
base_results = eval_model(base_model)

  0%|          | 0/500 [00:00<?, ?it/s]

Starting sample 0 , debit_card_specializing


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
  0%|          | 1/500 [00:21<2:57:51, 21.39s/it]

Finished Generating
Finished Execution

Starting sample 1 , debit_card_specializing
Finished Generating


  0%|          | 2/500 [00:53<3:49:49, 27.69s/it]

...

Finished Generating
Sample 497: SQLite error: no such column: T1.card_id
Finished Execution

Starting sample 498 , financial


100%|█████████▉| 499/500 [1:39:25<00:12, 12.82s/it]

Finished Generating
Finished Execution

Starting sample 499 , financial


100%|██████████| 500/500 [1:39:37<00:00, 11.96s/it]

Finished Generating
Finished Execution



In [18]:
print(base_results[0])
print(base_results[1])

{'correct_output': 0.458, 'exact_sql': 0.19, 'sql_errors': 0.3}
{'syntax': 7, 'no_column': 121, 'misuse': 4, 'timeouts': 12}


In [20]:
connections.clear()
lora_results = eval_model(lora_model)

  0%|          | 0/500 [00:00<?, ?it/s]

Starting sample 0 , debit_card_specializing


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
  0%|          | 1/500 [00:20<2:54:35, 20.99s/it]

Finished Generating
Finished Execution

Starting sample 1 , debit_card_specializing
Finished Generating


  0%|          | 2/500 [00:51<3:42:59, 26.87s/it]

...

Finished Generating
Sample 497: SQLite error: no such column: T1.card_id
Finished Execution

Starting sample 498 , financial


100%|█████████▉| 499/500 [1:30:46<00:12, 12.84s/it]

Finished Generating
Finished Execution

Starting sample 499 , financial


100%|██████████| 500/500 [1:30:58<00:00, 10.92s/it]

Finished Generating
Finished Execution



In [21]:
print(lora_results[0])
print(lora_results[1])

{'correct_output': 0.466, 'exact_sql': 0.194, 'sql_errors': 0.278}
{'syntax': 7, 'no_column': 122, 'misuse': 4, 'timeouts': 0}
